In [34]:
from pathlib import Path
import pandas as pd
import numpy as np
# Define the path of your folder containing 60 zips
zip_folder = Path(r"C:\Users\ASUS\Desktop\tennis_data")

# Read proper tables
MatchVotesInfo_df = pd.read_parquet(zip_folder / "votes_all.parquet")

MatchEventInfo_df = pd.read_parquet(zip_folder / "event_all.parquet")

# From MatchEventInfo, we only want the columns winner_code and match_id to enable merging!
# Then we drop duplicates and NaN values!
MatchEventInfo_df = MatchEventInfo_df[["match_id","winner_code"]]
MatchEventInfo_df.drop_duplicates(inplace=True)
MatchEventInfo_df.dropna(subset="winner_code", inplace=True)

# In MatchVotes, we drop date column to enable dropping duplicates
# The tables are actually snapshots of data, so it is convenient to keep the last data.
MatchVotesInfo_df.drop(columns="date",inplace=True)
MatchVotesInfo_df.drop_duplicates("match_id",keep="last",inplace=True)

# Now, it is time to merge data
# We use inner join to avoid creating NaN values!
result_df = pd.merge(MatchEventInfo_df, MatchVotesInfo_df, on="match_id", how="inner")

In [ ]:
# Now, let's create some useful columns
result_df["total_votes"] = (result_df["home_vote"] + result_df["away_vote"])

result_df["winner_votes"] = np.where( result_df["winner_code"] == 1, result_df["home_vote"], result_df["away_vote"])

result_df["loser_votes"] = np.where(result_df["winner_code"] == 1,result_df["away_vote"],result_df["home_vote"])

result_df["winner_fan_accuracy"] = (result_df["winner_votes"] /result_df["total_votes"] * 100)

result_df["loser_fan_accuracy"] = (result_df["loser_votes"] /result_df["total_votes"] * 100)

# Now, drop NaN values from accuracy columns and filter data
result_df.dropna(subset=["winner_fan_accuracy","loser_fan_accuracy"], inplace=True)
result_df.sort_values(by="winner_fan_accuracy", ascending=False, inplace=True)
result_df = result_df[(result_df["home_vote"] >= 20) & (result_df["away_vote"] >= 20)]

# Show result:
result_df.head(20)